In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os, json
from tqdm import tqdm
import time
import re
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np




BASE_URL = "https://law.justia.com"
COURTS = {
    #"supreme": "supreme-court",
    "appellate": [
        "court-of-appeals-first-appellate-district",
        "court-of-appeals-second-appellate-district",
        "court-of-appeals-third-appellate-district",
        "court-of-appeals-fourth-appellate-district",
        "court-of-appeals-fifth-appellate-district"]
}
START_YEAR = 2012
END_YEAR = 2024
SAVE_DIR = "illinois_med_mal_cases"

HEADERS = {"User-Agent": "LegalBot/1.0"}

KEYWORDS = [
    "malpractice", "medical negligence", "standard of care",
    "informed consent", "physician", "doctor", "hospital", "surgery",
    "surgical error", "diagnosis", "treatment", "healthcare provider",
    "duty of care"
]


C:\Users\Kyle\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
def contains_med_mal(text):
    lowered = text.lower()
    return any(kw in lowered for kw in KEYWORDS)

In [9]:
def get_soup(url):
    time.sleep(0.5)  # polite crawling
    #headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

def get_circuit_links():
    soup = get_soup(INDEX_URL)
    return [urljoin(BASE_URL, a['href']) for a in soup.select("a[href^='/cases/federal/appellate-courts/ca']")]

def crawl_case_links(court_type, year):
    base = f"{BASE_URL}/cases/illinois/{court_type}/{year}/"
    try:
        soup = get_soup(base)
        return [urljoin(BASE_URL, a['href']) for a in soup.select("a[href^='/cases/illinois']") if a['href'].endswith(".html")]
    except Exception:
        return []


In [10]:
def extract_case_data(case_url, court, year):
    try:
        soup = get_soup(case_url)
        title = soup.find("h1").get_text(strip=True)
        opinion_div = soup.find("div", {"id": "opinion"})
        if not opinion_div:
            return None
        case_text = opinion_div.get_text(separator="\n", strip=True)
        if not contains_med_mal(case_text):
            return None
        return {
            "title": title,
            "court": f"Illinois {court.capitalize()} Court",
            "year": year,
            "url": case_url,
            "text": case_text
        }
    except Exception:
        return None

In [11]:
def save_case_json(case_data):
    os.makedirs(SAVE_DIR, exist_ok=True)
    filename = case_data["title"].replace(" ", "_").replace("/", "-")[:100]
    filename = f"{case_data['court'].split()[-1]}_{case_data['year']}_{filename}.json"
    path = os.path.join(SAVE_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(case_data, f, indent=2)

In [18]:
def run_crawler():
    for court_key, court_path in COURTS.items():
        if court_key == "appellate":
            for district in court_path:
                for year in range(START_YEAR, END_YEAR + 1):
                    case_links = crawl_case_links(district, year)
                    for url in tqdm(case_links, desc=f"{district} {year}"):
                        case_data = extract_case_data(url, district, year)
                        if case_data:
                            save_case_json(case_data)
        else:
            for year in range(START_YEAR, END_YEAR + 1):
                case_links = crawl_case_links(court_path, year)
                for url in tqdm(case_links, desc=f"{court_key} {year}"):
                    case_data = extract_case_data(url, court_key, year)
                    if case_data:
                        save_case_json(case_data)
run_crawler()

court-of-appeals-fifth-appellate-district 2024: 100%|██████████████████████████████████| 36/36 [00:38<00:00,  1.07s/it]


In [4]:
import os

folder = "illinois_med_mal_cases"
for file in os.listdir(folder):
    path = os.path.join(folder, file)
    if os.path.getsize(path) == 0:
        print(f"Deleting empty file: {file}")
        os.remove(path)

Deleting empty file: Court_2012_In_re
Deleting empty file: Court_2013_In_re
Deleting empty file: Court_2014_In_re
Deleting empty file: Court_2015_In_re
Deleting empty file: Court_2023_In_re


In [2]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = []

for case_file in os.listdir("illinois_med_mal_cases"):
    with open(os.path.join("illinois_med_mal_cases", case_file), "r", encoding="utf-8") as f:
        case = json.load(f)
        splits = splitter.split_text(case["text"])
        for i, chunk in enumerate(splits):
            chunks.append({
                "title": case["title"],
                "year": case["year"],
                "court": case["court"],
                "source": case["url"],
                "content": chunk
            })

In [3]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c["content"] for c in chunks]
vectors = embedder.encode(texts, show_progress_bar=True)

Batches:   0%|                                                                               | 0/14778 [00:00<?, ?it/s]C:\Users\Kyle\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Batches: 100%|███████████████████████████████████████████████████████████████████| 14778/14778 [31:27<00:00,  7.83it/s]


In [4]:
dim = len(vectors[0])
index = faiss.IndexFlatL2(dim)
index.add(np.array(vectors))

In [5]:
def get_top_chunks(query, k=5):
    query_vec = embedder.encode([query])
    D, I = index.search(np.array(query_vec), k)
    return [chunks[i] for i in I[0]]

In [22]:
query = "What is standard of care for endoscopic retrograde cholangiopancreatography (ERCP)?"
context_chunks = get_top_chunks(query, k = 5)
citations = [
    f"{chunk['title']} ({chunk['year']}, {chunk['court']})\n{chunk['source']}"
    for chunk in context_chunks
]

context = "\n\n".join(c["content"] for c in context_chunks)
prompt = f"""You are a legal assistant. Answer the following question based on case law context below. Include citations.

Context:
{context}

Citations:
{chr(10).join(citations)}

Question:
{query}

Answer:"""



In [23]:
import openai
import os
api_key = os.getenv("4061_API_KEY")
client = openai.OpenAI(api_key = api_key)
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{'role':'user', 'content':prompt}])

print(response.choices[0].message.content)

The standard of care regarding endoscopic retrograde cholangiopancreatography (ERCP) is that it should be performed when there are signs of complications such as bile duct obstruction, particularly when a diagnosis of a mass or potential cancer is considered. In the context of the presented case law, it is deemed essential for a physician to refer a patient for an ERCP when they encounter symptoms or diagnostic imaging indicating significant concerns, such as a dilated pancreatic duct or bile duct blockage. 

For instance, in *Perkey v. Portes-Jarol*, the court noted that a reasonable family practice physician should have made an immediate referral to a gastroenterologist once the pancreatic duct was found to be three times its normal size, and the absence of such a referral was considered a breach of the standard of care. Specifically, it was established that Dr. Portes failed to act appropriately by not ensuring the earliest possible assessment for potential cancer or stricture affec